In [1]:
import torch
from torch.distributions import Normal

In [3]:


# 1. Setup Batched Inputs
batch_size = 5
S = torch.tensor([90.0, 95.0, 100.0, 105.0, 110.0], requires_grad=True)
K = torch.tensor([100.0, 100.0, 100.0, 100.0, 100.0])
T = torch.tensor([1.0], requires_grad=True)
r = torch.tensor([0.05])
sigma = torch.tensor([0.2], requires_grad=True)

dist = Normal(0, 1)

# 2. PyTorch Autograd Calculation
d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
d2 = d1 - sigma * torch.sqrt(T)
price = S * dist.cdf(d1) - K * torch.exp(-r * T) * dist.cdf(d2)

ones = torch.ones_like(price)
# Delta: dPrice/dS
delta_autograd = torch.autograd.grad(price, S, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
# Gamma: dDelta/dS
gamma_autograd = torch.autograd.grad(delta_autograd, S, grad_outputs=ones, retain_graph=True)[0]
# Vega: dPrice/dSigma
vega_autograd = torch.autograd.grad(price, sigma, grad_outputs=ones, retain_graph=True)[0]

# 3. Analytical Formulas
# Delta = N(d1)
delta_theory = dist.cdf(d1)

# Gamma = N'(d1) / (S * sigma * sqrt(T))
pdf_d1 = torch.exp(dist.log_prob(d1))
gamma_theory = pdf_d1 / (S * sigma * torch.sqrt(T))

# Vega = S * N'(d1) * sqrt(T)
# (Note: Autograd gives total derivative; divide by 100 for '1% vol move' convention)
vega_theory = S * pdf_d1 * torch.sqrt(T)

# 4. Verification Results
print(f"{'Spot':<8} | {'Delta Err':<12} | {'Gamma Err':<12} | {'Vega Err':<12}")
print("-" * 55)
for i in range(batch_size):
    d_err = (delta_autograd[i] - delta_theory[i]).abs().item()
    g_err = (gamma_autograd[i] - gamma_theory[i]).abs().item()
    v_err = (vega_autograd.sum() - vega_theory.sum()).abs().item() # Vega is scalar-based here
    print(f"{S[i].item():<8.2f} | {d_err:<12.2e} | {g_err:<12.2e} | {v_err:<12.2e}")

Spot     | Delta Err    | Gamma Err    | Vega Err    
-------------------------------------------------------
90.00    | 2.09e-07     | 1.86e-09     | 3.05e-05    
95.00    | 0.00e+00     | 0.00e+00     | 3.05e-05    
100.00   | 5.96e-07     | 5.59e-09     | 3.05e-05    
105.00   | 0.00e+00     | 0.00e+00     | 3.05e-05    
110.00   | 5.96e-08     | 5.59e-09     | 3.05e-05    


In [4]:
def black_scholes_call(S, K, T, r, sigma):
    # Standard Black-Scholes formula
    # S: Spot, K: Strike, T: Time to maturity, r: risk-free rate, sigma: volatility
    dist = Normal(0, 1)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    price = S * dist.cdf(d1) - K * torch.exp(-r * T) * dist.cdf(d2)
    return price, d1, dist

# 1. Setup Inputs
S = torch.tensor([100.0], requires_grad=True)
K = torch.tensor([100.0])
T = torch.tensor([1.0], requires_grad=True)
r = torch.tensor([0.05], requires_grad=True)
sigma = torch.tensor([0.2], requires_grad=True)

# 2. Compute Price
price, d1, dist = black_scholes_call(S, K, T, r, sigma)

# 3. Compute First-Order Greeks (Delta, Vega, Theta)
# We use autograd.grad to keep the graph alive for Gamma later
delta = torch.autograd.grad(price, S, create_graph=True)[0]
vega = torch.autograd.grad(price, sigma, retain_graph=True)[0]
theta = -torch.autograd.grad(price, T, retain_graph=True)[0] # Usually expressed as negative

# 4. Compute Second-Order Greek (Gamma)
gamma = torch.autograd.grad(delta, S)[0]

# --- Verification Logic ---
# Theoretical Delta: N(d1)
theoretical_delta = dist.cdf(d1)
# Theoretical Gamma: N'(d1) / (S * sigma * sqrt(T))
theoretical_gamma = torch.exp(dist.log_prob(d1)) / (S * sigma * torch.sqrt(T))

print(f"Option Price: {price.item():.4f}")
print("-" * 30)
print(f"Delta (Autograd): {delta.item():.4f} | Theoretical: {theoretical_delta.item():.4f}")
print(f"Gamma (Autograd): {gamma.item():.4f} | Theoretical: {theoretical_gamma.item():.4f}")
print(f"Vega  (Autograd): {vega.item():.4f}")
print(f"Theta (Autograd): {theta.item():.4f}")

Option Price: 10.4506
------------------------------
Delta (Autograd): 0.6368 | Theoretical: 0.6368
Gamma (Autograd): 0.0188 | Theoretical: 0.0188
Vega  (Autograd): 37.5241
Theta (Autograd): -6.4140
